# AIND Ephys Spike Sorting (Kilosort4)

Build an `AINDEPhysSpikesortKilosort4ScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysSpikesortKilosort4Task` for each coordinate. The task itself clones [aind-ephys-spikesort-kilosort4](https://github.com/AllenNeuralDynamics/aind-ephys-spikesort-kilosort4) on first use, seeds its `data/` with the `preprocessed_<name>/` directories produced by the preprocessing stage, writes a `params_obi.json`, runs `code/run_capsule.py`, and copies the resulting `spikesorted_<name>/` into the single config's `coordinate_output_root` (`obi-output/03_aind_ephys_spikesort_kilosort4/grid_scan/0/`).

This notebook passes `preprocessing_output_path=None`, so the task **materializes its input on the fly** — it synthesises a toy recording and runs it through the dispatch and preprocessing stages (clipped to 9 s, motion off, long enough for KS4 to fit whitening / templates) before sorting. That makes the notebook runnable standalone, without first running notebooks 00–02. We just tune a few sorter knobs (`whitening_range`, `nskip`, `nearest_chans`, `nearest_templates`, `do_correction=False`, `nblocks=0`, `torch_device="cpu"`) for the small toy probe.

## Imports and prerequisites

In [1]:
import subprocess
import sys
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.aind_ephys._03_kilosort4.blocks import Kilosort4Sorter

In [2]:
# The Kilosort4 capsule runs in its own isolated virtual environment, built
# automatically by the task from the capsule's dependencies -- kilosort, torch,
# spikeinterface, etc. (the first run builds it under
# /tmp/aind-ephys-spikesort-kilosort4/.obi-capsule-venv and may take a minute).
# obi-one's environment is therefore left untouched -- no aind-data-schema, no
# pydantic downgrade. The only thing this kernel needs is spikeinterface, to load
# and inspect the sorting result in the final cell.
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "spikeinterface==0.104.7",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Checked 1 package in 3ms


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'spikeinterface==0.104.7'], returncode=0)

## Build the spike-sorting scan config

With `preprocessing_output_path=None`, the task **materializes everything it needs on the fly** — it synthesises a toy recording and runs it through the dispatch and preprocessing stages — so this notebook runs standalone. Set it to an existing preprocessing output directory (e.g. notebook 02's `coordinate_output_root`) to sort that instead.

In [ ]:
# preprocessing_output_path=None makes the task materialize everything it needs on
# the fly -- a toy recording run through the dispatch + preprocessing stages -- so
# this notebook runs standalone, without first running notebooks 00-02. To sort an
# existing preprocessing output instead, set it to that directory, e.g.
# (Path.cwd() / "../../../obi-output/02_aind_ephys_preprocessing/grid_scan/0").resolve()
ks_scan = obi.AINDEPhysSpikesortKilosort4ScanConfig(
    initialize=obi.AINDEPhysSpikesortKilosort4ScanConfig.Initialize(
        preprocessing_output_path=None,
        skip_motion_correction=True,
        min_drift_channels=64,
        n_jobs=1,
    ),
    sorter=Kilosort4Sorter(
        do_correction=False,
        nblocks=0,
        torch_device="cpu",
        whitening_range=16,
        nskip=1,
        nearest_chans=8,
        nearest_templates=16,
    ),
)

## Generate the grid scan and run the sorting task

In [4]:
ks_grid = obi.GridScanGenerationTask(
    form=ks_scan,
    output_root="../../../obi-output/03_aind_ephys_spikesort_kilosort4/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
ks_grid.execute()

obi.run_tasks_for_generated_scan(ks_grid)

[2026-06-30 15:53:50,137] INFO: Seeded 1 preprocessed recording(s) into /tmp/aind-ephys-spikesort-kilosort4/data
[2026-06-30 15:53:50,141] INFO: Running /tmp/aind-ephys-spikesort-kilosort4/.obi-capsule-venv/bin/python -u run_capsule.py --params params_obi.json --n-jobs 1 --min-drift-channels 64 --skip-motion-correction --raise-if-fails
[2026-06-30 15:54:00,461] INFO: 

SPIKE SORTING WITH KILOSORT4

	RAISE_IF_FAILS: True
	SKIP_MOTION_CORRECTION: True
	MIN_DRIFT_CHANNELS: 64
	N_JOBS: -1
Sorting recording: block0_None_recording1
BinaryFolderRecording: 70 channels - 30.0kHz - 1 segments - 270,001 samples - 9.00s 
                       float32 dtype - 72.10 MiB
Drift correction disabled
	Raw sorting output: KiloSortSortingExtractor: 14 units - 1 segments - 30.0kHz
	Sorting output without empty units: UnitsSelectionSorting: 14 units - 1 segments - 30.0kHz
	Saving results to ../results/spikesorted_block0_None_recording1
SPIKE SORTING time: 9.39s



[PosixPath('../../../obi-output/03_aind_ephys_spikesort_kilosort4/grid_scan/0')]

## Load the sorting result

In [5]:
import spikeinterface.full as si

coord_dir = Path(ks_grid.single_configs[0].coordinate_output_root)
print("coordinate_output_root:", coord_dir)

sorting_dirs = [
    p for p in coord_dir.iterdir() if p.is_dir() and p.name.startswith("spikesorted_")
]
print("Sorting dirs:", [p.name for p in sorting_dirs])

if sorting_dirs:
    sorting = si.load(sorting_dirs[0])
    print(sorting)
    print("num_units:", len(sorting.unit_ids))
    print("unit_ids:", list(sorting.unit_ids))

coordinate_output_root: ../../../obi-output/03_aind_ephys_spikesort_kilosort4/grid_scan/0
Sorting dirs: ['spikesorted_block0_None_recording1']
NumpyFolder (NumpyFolderSorting): 14 units - 1 segments - 30.0kHz
num_units: 14
unit_ids: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
